# DistilBERT Fine-tuning for Phishing Email Detection
### PROM02 Dissertation — Objective O2 (model 3 of 3: fine-tuned DistilBERT)

This notebook trains and evaluates the DistilBERT model, completing the three-way comparison alongside the
Logistic Regression and Random Forest baselines already trained locally (see `tables/T11_baseline_model_comparison.csv`).

**Before running**: Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4 is fine).

**Resumability**: every heavy step (dataset download/cleaning, model checkpoints) is cached to Google Drive.
If Colab disconnects for any reason, just reconnect and Runtime -> Run all again. Cached steps skip instantly,
and training resumes from the last saved checkpoint rather than starting over.

## 1. Check GPU

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. Go to Runtime > Change runtime type > GPU, then re-run this cell.')

CUDA available: True
GPU: Tesla T4


## 2. Mount Google Drive

All cleaned data, checkpoints and the final model are stored under `DRIVE_PROJECT_DIR` so they survive a
disconnect or a fresh runtime.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/prom02_phishing_pdat'
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT_DIR, 'distilbert_checkpoints')
FINAL_MODEL_DIR = os.path.join(DRIVE_PROJECT_DIR, 'distilbert_final_model')
DATA_CACHE_DIR = os.path.join(DRIVE_PROJECT_DIR, 'data_cache')
RESULTS_DIR = os.path.join(DRIVE_PROJECT_DIR, 'results')

for d in [DRIVE_PROJECT_DIR, CHECKPOINT_DIR, FINAL_MODEL_DIR, DATA_CACHE_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('Project directory ready at:', DRIVE_PROJECT_DIR)

Mounted at /content/drive
Project directory ready at: /content/drive/MyDrive/prom02_phishing_pdat


## 3. Install dependencies

In [3]:
!pip install -q kagglehub transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


## 4. Load and prepare the dataset

This reproduces the *exact* cleaning and split logic used locally in `src/data/prepare_dataset.py`
(same thresholds, same `random_state=42`), so the DistilBERT test set matches the baselines' test set
and the three-way comparison is fair. Results are cached to Drive so this only runs once, even across
disconnects.

In [4]:
import pandas as pd
import re
import glob
from pathlib import Path

TRAIN_CACHE = os.path.join(DATA_CACHE_DIR, 'train.csv')
VAL_CACHE = os.path.join(DATA_CACHE_DIR, 'val.csv')
TEST_CACHE = os.path.join(DATA_CACHE_DIR, 'test.csv')

MIN_CHARS = 10
MAX_CHARS = 50000
RANDOM_STATE = 42
TEXT_COL = 'text_combined'
LABEL_COL = 'label'

def clean_text(s):
    if not isinstance(s, str):
        return ''
    s = s.replace('\x00', '')
    s = re.sub(r'[\r\n\t]+', ' ', s)
    s = re.sub(r' {2,}', ' ', s)
    return s.strip()

if os.path.exists(TRAIN_CACHE) and os.path.exists(VAL_CACHE) and os.path.exists(TEST_CACHE):
    print('Found cached splits on Drive, loading those (skipping download/cleaning).')
    train_df = pd.read_csv(TRAIN_CACHE)
    val_df = pd.read_csv(VAL_CACHE)
    test_df = pd.read_csv(TEST_CACHE)
else:
    print('No cached splits found. Downloading dataset from Kaggle...')
    import kagglehub
    path = kagglehub.dataset_download('naserabdullahalam/phishing-email-dataset')
    print('Path to dataset files:', path)

    csv_candidates = glob.glob(os.path.join(path, '**', 'phishing_email.csv'), recursive=True)
    assert csv_candidates, f'Could not find phishing_email.csv under {path}'
    raw_csv = csv_candidates[0]
    print('Using combined dataset file:', raw_csv)

    df = pd.read_csv(raw_csv)
    n_start = len(df)
    df = df.drop_duplicates()
    df[TEXT_COL] = df[TEXT_COL].apply(clean_text)
    lengths = df[TEXT_COL].str.len()
    df = df[~((lengths < MIN_CHARS) | (lengths > MAX_CHARS))].copy()
    print(f'Cleaned dataset: {n_start} -> {len(df)} rows')

    from sklearn.model_selection import train_test_split
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df[LABEL_COL], random_state=RANDOM_STATE)
    val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df[LABEL_COL], random_state=RANDOM_STATE)

    train_df.to_csv(TRAIN_CACHE, index=False)
    val_df.to_csv(VAL_CACHE, index=False)
    test_df.to_csv(TEST_CACHE, index=False)
    print('Cached splits saved to Drive for future runs.')

print(f'train={len(train_df)} val={len(val_df)} test={len(test_df)}')
train_df[LABEL_COL].value_counts(normalize=True)

No cached splits found. Downloading dataset from Kaggle...
Using Colab cache for faster access to the 'phishing-email-dataset' dataset.
Path to dataset files: /kaggle/input/phishing-email-dataset
Using combined dataset file: /kaggle/input/phishing-email-dataset/phishing_email.csv
Cleaned dataset: 82486 -> 82016 rows
Cached splits saved to Drive for future runs.
train=57411 val=12302 test=12303


,proportion
label,
1,0.522182
0,0.477818


## 5. Tokenisation

Emails are truncated to 512 tokens (DistilBERT's maximum). Most emails are well under this after cleaning
(EDA found a 99th-percentile length of ~9,000 characters, roughly 1,500-2,000 tokens), but longer emails will
have their tail cut off. This truncation is worth noting explicitly as a limitation in the dissertation.

In [5]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    return Dataset.from_pandas(df[[TEXT_COL, LABEL_COL]].rename(columns={LABEL_COL: 'labels'}), preserve_index=False)

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)

def tokenize_fn(batch):
    return tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LENGTH, padding='max_length')

train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=[TEXT_COL])
val_ds = val_ds.map(tokenize_fn, batched=True, remove_columns=[TEXT_COL])
test_ds = test_ds.map(tokenize_fn, batched=True, remove_columns=[TEXT_COL])

print(train_ds)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/57411 [00:00<?, ? examples/s]

Map:   0%|          | 0/12302 [00:00<?, ? examples/s]

Map:   0%|          | 0/12303 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 57411
})


## 6. Model and training configuration

In [7]:
import numpy as np
from scipy.special import softmax
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=1)[:, 1]
    preds = np.argmax(logits, axis=1)
    return {
        'precision': precision_score(labels, preds),
        'recall': recall_score(labels, preds),
        'f1': f1_score(labels, preds),
        'auc_roc': roc_auc_score(labels, probs),
    }

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to=[],
    seed=42,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 7. Train (safe to re-run after a disconnect)

This cell automatically detects the latest saved checkpoint in `CHECKPOINT_DIR` and resumes from it if one
exists, rather than starting over.

In [8]:
from transformers.trainer_utils import get_last_checkpoint

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR) if os.path.isdir(CHECKPOINT_DIR) else None
if last_checkpoint:
    print('Resuming from checkpoint:', last_checkpoint)
else:
    print('No checkpoint found, starting fresh.')

trainer.train(resume_from_checkpoint=last_checkpoint)

No checkpoint found, starting fresh.


Step,Training Loss,Validation Loss,Precision,Recall,F1,Auc Roc
500,0.078647,0.131220,0.939434,0.992372,0.965178,0.996325
1000,0.094836,0.075238,0.992675,0.970423,0.981423,0.998369
1500,0.063146,0.063824,0.994751,0.973537,0.984030,0.999037
2000,0.056438,0.033903,0.995743,0.983032,0.989347,0.999293
2500,0.054014,0.032222,0.991265,0.989259,0.990261,0.999441
3000,0.043669,0.036363,0.983584,0.997976,0.990728,0.999627
3500,0.030474,0.027991,0.992698,0.994707,0.993702,0.999665
4000,0.009715,0.044818,0.995457,0.989103,0.992270,0.999343
4500,0.010317,0.039226,0.992094,0.996264,0.994175,0.999098
5000,0.001530,0.034724,0.994396,0.994396,0.994396,0.999681


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10767, training_loss=0.028354798625347462, metrics={'train_runtime': 3466.6729, 'train_samples_per_second': 49.683, 'train_steps_per_second': 3.106, 'total_flos': 2.281525747271885e+16, 'train_loss': 0.028354798625347462, 'epoch': 3.0})

## 8. Evaluate on the held-out test set

In [9]:
test_results = trainer.evaluate(test_ds, metric_key_prefix='test')
print(test_results)

result_row = pd.DataFrame([{
    'model': 'DistilBERT (fine-tuned)',
    'precision': round(test_results['test_precision'], 4),
    'recall': round(test_results['test_recall'], 4),
    'f1_score': round(test_results['test_f1'], 4),
    'auc_roc': round(test_results['test_auc_roc'], 4),
}])
result_row.to_csv(os.path.join(RESULTS_DIR, 'T12_distilbert_result.csv'), index=False)
print('Saved result row to Drive:', os.path.join(RESULTS_DIR, 'T12_distilbert_result.csv'))
result_row

Training Loss,Validation Loss,Step,Precision,Recall,F1,Auc Roc
0.000026,0.029389,10767,0.995640,0.995330,0.995485,0.999751


{'test_loss': 0.029388995841145515, 'test_precision': 0.9956399875428216, 'test_recall': 0.9953300124533001, 'test_f1': 0.9954849758679745, 'test_auc_roc': 0.9997508783929631}
Saved result row to Drive: /content/drive/MyDrive/prom02_phishing_pdat/results/T12_distilbert_result.csv


,model,precision,recall,f1_score,auc_roc
0,DistilBERT (fine-tuned),0.9956,0.9953,0.9955,0.9998


## 9. Save the final model and tokenizer

Saved to Drive so it can be downloaded and embedded in the local Flask PDAT tool later.

In [10]:
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print('Final model saved to:', FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to: /content/drive/MyDrive/prom02_phishing_pdat/distilbert_final_model


## 10. Bringing results back to the local project

After this notebook finishes:
1. Download `T12_distilbert_result.csv` from `MyDrive/prom02_phishing_pdat/results/` and place it in the local `Dissertation/tables/` folder.
2. Download the `distilbert_final_model` folder from Drive and place it in the local `Dissertation/models/` folder,
   for use later when building the Flask PDAT tool and its LIME/SHAP explainability layer.
3. Merge `T11_baseline_model_comparison.csv` and `T12_distilbert_result.csv` into one three-row comparison table
   for the Results chapter.